# Gold Rod Heat Diffusion — PINN vs. FEM

This notebook solves the 1D heat equation for a gold rod over a **3600-second**
window, with:
- a heater fixed at `T_LEFT` at `x = 0` (Dirichlet)
- an **insulated** end at `x = L` (Neumann, zero flux)
- a cold rod initially at `T_INIT`

Two independent solvers are built and compared:
1. **PINN** — trained with a time curriculum (grows the training window from
   60s to the full 3600s) using Adam only, checkpointing its best weights
   throughout. L-BFGS is intentionally left out of this version -- see the
   note in section 2.3b for why.
2. **FEM** — a classical finite element solver, used as ground truth.

> **Note on expectations:** a PINN is a trained approximation, not an exact
> solve. "Extremely low error" here means single-digit-percent relative
> error against the FEM ground truth, not exact zero — driving it to
> literal zero isn't achievable for any trained network. The final cell
> reports both absolute and relative error so you can judge this honestly.

## 1. Setup

In [ ]:
import copy
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.linalg import lu_factor, lu_solve

torch.manual_seed(0)
np.random.seed(0)

### Physical parameters

`T_MAX` is fixed at its true target value (3600s) from the very start.
This matters even though training uses a curriculum that starts on a
shorter window -- the model's `t_norm = t / T_MAX` normalization must be
consistent for the network's entire life, or anything it learns early
becomes meaningless once the window grows.

In [ ]:
L = 1.0           # rod length (m)
T_MAX = 3600.0    # total simulated time (s) -- the real target duration
alpha = 1.22e-3    # thermal diffusivity (m^2/s)

T_INIT = 20.0      # initial rod temperature (deg C), applied at t = 0 for all x
T_LEFT = 100.0     # boundary condition at x = 0, held for all t (Dirichlet / heater)
# x = L is insulated: zero heat flux (Neumann), no fixed value

## 2. PINN Solver

### 2.1 Model definition

A Fourier feature embedding fights spectral bias (the tendency of plain
tanh MLPs to only learn smooth, low-frequency functions, which previously
collapsed training into a low-varying quadratic). `scale=2.0` was tuned
down from an initial `10.0` after a higher scale let L-BFGS overfit and
produce oscillatory, physically nonsensical output between collocation
points.

In [ ]:
class FourierFeatures(nn.Module):
    def __init__(self, num_features=32, scale=2.0):
        super().__init__()
        # fixed (non-trainable) random projection -- this is what lets the
        # network represent sharp, high-frequency features
        self.B = nn.Parameter(torch.randn(2, num_features) * scale, requires_grad=False)

    def forward(self, xt):
        proj = xt @ self.B
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)


class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.fourier = FourierFeatures(num_features=32, scale=2.0)
        self.net = nn.Sequential(
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, x, t):
        # normalize t against the TRUE T_MAX (3600), always -- even during
        # the early curriculum stages when we only sample t up to 60s.
        # This keeps t_norm's meaning fixed for the model's entire life.
        t_norm = t / T_MAX
        inp = torch.stack((x, t_norm), dim=1)
        feat = self.fourier(inp)
        return self.net(feat)


model = PINN()
model

### 2.2 Training configuration

In [ ]:
N_F = 3000    # interior (physics) points per epoch
N_IC = 300    # initial-condition points
N_BC = 300    # boundary points per side

adam_epochs = 10000
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2000, gamma=0.5)

w_physics, w_ic, w_bc = 7.0, 1.0, 4.0

# ---- curriculum schedule ----
# training starts on an easy, well-posed 60-second window (matches the
# domain size that was known to train reliably) and grows to the full
# T_MAX over the first 60% of Adam epochs. The remaining 40% trains on
# the complete domain to consolidate.
initial_t_max = 60.0
curriculum_fraction = 0.6

grad_clip_norm = 5.0

### 2.3 Stage 1 — Adam training loop (with curriculum)

Key properties of this loop:
- **Curriculum growth.** `current_t_max` starts at `initial_t_max` and
  linearly grows to `T_MAX` by `curriculum_fraction` of the way through
  training. Both the interior (physics) sampling *and* the boundary
  sampling grow together -- the network is never asked to satisfy a
  boundary condition on a time range it hasn't practiced the interior
  physics on yet.
- **Three-way stratified spatial sampling.** Split evenly between: uniform
  coverage of `[0, L]`, concentrated near `x=0` (the heater's sharp boundary
  layer), and concentrated near `x=L` (the insulated wall, which develops
  its own local curvature once the diffusion front reaches it). The
  `x=L` bias was added after a run showed a large, localized error pocket
  right where the far wall starts accumulating heat -- that region wasn't
  getting dedicated training density before.
- **IC sampling excludes a tiny neighborhood of `x=0`**, avoiding the
  discontinuous `(x=0, t=0)` corner.
- **Gradient clipping** caps how large a single update can be -- more
  important on a domain this wide, where occasional bad batches can
  otherwise produce a destabilizing step.
- **Best-weights checkpointing.** Whenever total loss hits a new low, the
  model's weights are saved. After training, the best checkpoint (not
  necessarily the final epoch) is reloaded -- this protects the final
  result from a late unlucky epoch.

In [ ]:
loss_history = {"total": [], "physics": [], "ic": [], "bc": [], "current_t_max": []}
best_loss = float("inf")
best_state = None

for epoch in range(adam_epochs):
    optimizer.zero_grad()

    # ---- curriculum: how much of the time domain we train on right now ----
    progress = min(1.0, epoch / (adam_epochs * curriculum_fraction))
    current_t_max = initial_t_max + progress * (T_MAX - initial_t_max)

    # ---- interior collocation points: stratified in both x and t ----
    # x is split three ways: uniform coverage, biased toward x=0 (heater's
    # sharp boundary layer), and biased toward x=L (the insulated wall,
    # which develops its own local curvature once the diffusion front
    # reaches it -- this third group was missing before, and was exactly
    # the gap that produced the large localized error near x~0.8, t~2000).
    n_third = N_F // 3

    x_f_uniform = torch.rand(n_third) * L
    u_rand_left = torch.rand(n_third)
    x_f_biased_left = L * (1 - torch.sqrt(1 - u_rand_left))
    u_rand_right = torch.rand(N_F - 2 * n_third)
    x_f_biased_right = L * torch.sqrt(u_rand_right)
    x_f = torch.cat([x_f_uniform, x_f_biased_left, x_f_biased_right]).clone().requires_grad_(True)

    n_half = N_F // 2
    t_f_uniform = torch.rand(n_half) * current_t_max
    t_f_biased = (torch.rand(N_F - n_half) ** 1.3) * current_t_max
    t_f = torch.cat([t_f_uniform, t_f_biased]).clone().requires_grad_(True)

    u_f = model(x_f, t_f)
    du_dx = torch.autograd.grad(u_f, x_f, torch.ones_like(u_f), create_graph=True)[0]
    d2u_dx2 = torch.autograd.grad(du_dx, x_f, torch.ones_like(du_dx), create_graph=True)[0]
    du_dt = torch.autograd.grad(u_f, t_f, torch.ones_like(u_f), create_graph=True)[0]

    res = du_dt - alpha * d2u_dx2
    physics_loss = torch.mean(res ** 2)

    # ---- initial condition: exclude a tiny neighborhood of x=0 ----
    x_ic = torch.rand(N_IC) * L
    x_ic = x_ic[x_ic > 0.02 * L]
    t_ic = torch.zeros_like(x_ic)
    u_ic = model(x_ic, t_ic)
    ic_loss = torch.mean((u_ic - T_INIT) ** 2)

    # ---- left boundary: Dirichlet, heater fixed at T_LEFT ----
    # sampled over the CURRENT curriculum window, not the full T_MAX
    x_bc1 = torch.zeros(N_BC)
    t_bc1 = torch.linspace(0, current_t_max, N_BC)
    bc1_loss = torch.mean((model(x_bc1, t_bc1) - T_LEFT) ** 2)

    # ---- right boundary: Neumann, insulated (zero flux -> du/dx = 0) ----
    x_bc2 = torch.full((N_BC,), L, requires_grad=True)
    t_bc2 = torch.linspace(0, current_t_max, N_BC)
    u_bc2 = model(x_bc2, t_bc2)
    du_dx_bc2 = torch.autograd.grad(u_bc2, x_bc2, torch.ones_like(u_bc2), create_graph=True)[0]
    bc2_loss = torch.mean(du_dx_bc2 ** 2)

    bc_loss = bc1_loss + bc2_loss

    # ---- total loss ----
    loss = w_physics * physics_loss + w_ic * ic_loss + w_bc * bc_loss

    if not torch.isfinite(loss):
        print(f"epoch {epoch:5d} | non-finite loss encountered, skipping step")
        continue

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
    optimizer.step()
    scheduler.step()

    loss_val = loss.item()
    loss_history["total"].append(loss_val)
    loss_history["physics"].append(physics_loss.item())
    loss_history["ic"].append(ic_loss.item())
    loss_history["bc"].append(bc_loss.item())
    loss_history["current_t_max"].append(current_t_max)

    if loss_val < best_loss:
        best_loss = loss_val
        best_state = copy.deepcopy(model.state_dict())

    if epoch % 500 == 0:
        print(f"epoch {epoch:5d} | window {current_t_max:7.1f}s | total {loss_val:.4f} | "
              f"physics {physics_loss.item():.4f} | ic {ic_loss.item():.4f} | "
              f"bc {bc_loss.item():.4f}")

# ---- restore best checkpoint from the Adam phase ----
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nRestored best Adam checkpoint, loss = {best_loss:.5f}")

### Why there's no L-BFGS phase in this version

Given time pressure, the L-BFGS refinement stage has been removed. It's the
single component in this whole pipeline that caused unpredictable failures
earlier -- a long run against one frozen sample overfit that exact sample
(aided by the Fourier layer's capacity for oscillation) and produced wild,
physically nonsensical output. It's genuinely useful for squeezing out the
last bit of error once things are stable, but it's not worth the risk right
now. Adam alone, with the curriculum and checkpointing already in place, is
the reliable path -- more epochs, not a fancier optimizer, is the safe way
to lower error under time pressure.

To add L-BFGS back later for extra polish: run it as several short outer
passes (10 passes x 50 iterations, not one long 500-iteration run) with
fresh collocation points drawn each pass, and checkpoint the best result --
exactly the pattern used earlier in this conversation.

### 2.4 Training diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for key in ["total", "physics", "ic", "bc"]:
    axes[0].plot(loss_history[key], label=key)
axes[0].set_yscale("log")
axes[0].set_xlabel("epoch (Adam phase)")
axes[0].set_ylabel("loss (log scale)")
axes[0].legend()
axes[0].set_title("Adam-phase training loss components")

axes[1].plot(loss_history["current_t_max"])
axes[1].set_xlabel("epoch (Adam phase)")
axes[1].set_ylabel("current curriculum window (s)")
axes[1].set_title("Curriculum growth: training window over time")

plt.tight_layout()
plt.show()

### 2.5 PINN solution visualization

In [ ]:
n_plot = 120
x_lin = np.linspace(0, L, n_plot)
t_lin = np.linspace(0, T_MAX, n_plot)
x_grid, t_grid = np.meshgrid(x_lin, t_lin)

x_flat = torch.tensor(x_grid.flatten(), dtype=torch.float32)
t_flat = torch.tensor(t_grid.flatten(), dtype=torch.float32)

with torch.no_grad():
    u_grid = model(x_flat, t_flat).numpy().reshape(n_plot, n_plot)

print("u range:", u_grid.min(), "to", u_grid.max())

In [ ]:
fig = go.Figure(data=[go.Surface(
    x=x_lin,
    y=t_lin,
    z=u_grid,
    colorscale='Inferno',
    colorbar=dict(title='Temp (deg C)')
)])

fig.update_layout(
    title='PINN solution: temperature distribution in gold rod',
    scene=dict(
        xaxis_title='x (position along rod)',
        yaxis_title='t (time)',
        zaxis_title='u (temperature)',
        camera=dict(eye=dict(x=1.5, y=-1.8, z=0.9))
    ),
    width=800,
    height=600,
    margin=dict(l=0, r=0, t=40, b=0)
)

fig.show()

In [ ]:
plt.figure(figsize=(6, 5))
cp = plt.contourf(x_grid, t_grid, u_grid, levels=50, cmap='inferno')
plt.colorbar(cp, label='Temperature (deg C)')
plt.xlabel('x (position along rod)')
plt.ylabel('t (time)')
plt.title('PINN solution: temperature distribution in gold rod')
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
snapshot_times_plot = list(range(0, int(T_MAX) + 1, int(T_MAX // 12)))
for time_val in snapshot_times_plot:
    position_tensor = torch.linspace(0, L, 100)
    time_tensor = torch.full_like(position_tensor, float(time_val), dtype=torch.float32)

    with torch.no_grad():
        temp_output = model(position_tensor, time_tensor)
        temp_values = temp_output.squeeze().cpu().numpy()

    plt.plot(position_tensor.cpu().numpy(), temp_values, label=f"t = {time_val}s")

plt.xlabel('x (position along rod)')
plt.ylabel('u (temperature)')
plt.title('PINN solution at several times')
plt.legend(fontsize=8, ncol=2)
plt.grid(True)
plt.show()

## 3. FEM Reference Solution

Classical finite-element solve of the same problem, used as ground truth.

- **Left (`x=0`):** Dirichlet — clamped exactly to `T_LEFT` every time step.
- **Right (`x=L`):** Neumann, zero flux — FEM's *natural* boundary
  condition, requiring no explicit enforcement.
- **Time resolution scales with `T_MAX`** (`dt_target` fixed at 0.05s)
  rather than using a fixed step count, so accuracy doesn't degrade as the
  simulated duration grows.

In [ ]:
# ---- mesh ----
N_NODES = 101
dt_target = 0.05           # keep this time resolution regardless of T_MAX
n_dt = int(T_MAX / dt_target)
x_nodes = np.linspace(0, L, N_NODES)
h = x_nodes[1] - x_nodes[0]
dt = T_MAX / n_dt

print(f"FEM: {n_dt} time steps, dt = {dt:.4f}s")

# ---- assemble global mass (M) and stiffness (K) matrices ----
M = np.zeros((N_NODES, N_NODES))
K = np.zeros((N_NODES, N_NODES))

M_local = (h / 6.0) * np.array([[2, 1],
                                 [1, 2]])
K_local = (1.0 / h) * np.array([[1, -1],
                                 [-1, 1]])

for e in range(N_NODES - 1):
    nodes = [e, e + 1]
    for a in range(2):
        for b in range(2):
            M[nodes[a], nodes[b]] += M_local[a, b]
            K[nodes[a], nodes[b]] += K_local[a, b]

# ---- time discretization: implicit (backward) Euler ----
A = M + alpha * dt * K

# ---- left boundary (x=0): Dirichlet, clamp exactly ----
A[0, :] = 0.0
A[0, 0] = 1.0

# ---- right boundary (x=L): Neumann, zero flux ----
# natural boundary condition -- no modification needed

A_factored = lu_factor(A)

# ---- initial condition ----
T = np.full(N_NODES, T_INIT)
T[0] = T_LEFT

# ---- store snapshots for plotting ----
snapshot_times = list(range(0, int(T_MAX) + 1, int(T_MAX // 12)))
snapshots = {}
t_current = 0.0
next_snapshot_idx = 0

if 0 in snapshot_times:
    snapshots[0] = T.copy()
    next_snapshot_idx = 1

for step in range(n_dt):
    rhs = M @ T
    rhs[0] = T_LEFT

    T = lu_solve(A_factored, rhs)
    t_current += dt

    if (next_snapshot_idx < len(snapshot_times) and
            t_current >= snapshot_times[next_snapshot_idx] - 1e-6):
        snapshots[snapshot_times[next_snapshot_idx]] = T.copy()
        next_snapshot_idx += 1

plt.figure(figsize=(7, 5))
for t_val, T_snap in snapshots.items():
    plt.plot(x_nodes, T_snap, label=f"t = {t_val}s")

plt.xlabel("x (position along rod)")
plt.ylabel("Temperature (deg C)")
plt.title("FEM solution: insulated right end (Neumann BC)")
plt.legend(fontsize=8, ncol=2)
plt.grid(True)
plt.show()

## 4. Comparison: PINN vs. FEM

In [ ]:
plt.figure(figsize=(8, 6))
colors = plt.cm.inferno(np.linspace(0.15, 0.85, len(snapshots)))

for color, (t_val, T_snap) in zip(colors, snapshots.items()):
    plt.plot(x_nodes, T_snap, color=color, label=f"t = {t_val}s (FEM)")

    pinn_points = torch.tensor(x_nodes, dtype=torch.float32)
    pinn_time = torch.full_like(pinn_points, float(t_val), dtype=torch.float32)
    with torch.no_grad():
        pinn_temp = model(pinn_points, pinn_time).squeeze().numpy()
    plt.plot(x_nodes, pinn_temp, '--', color=color)

plt.plot([], [], 'k-', label='FEM (solid)')
plt.plot([], [], 'k--', label='PINN (dashed)')

plt.xlabel("x (position along rod)")
plt.ylabel("Temperature (deg C)")
plt.title("FEM (solid) vs. PINN (dashed) — insulated right end")
plt.legend(fontsize=8, ncol=2)
plt.grid(True)
plt.show()

In [ ]:
# ---- pointwise error over the full (x, t) grid ----
fem_grid = np.zeros_like(u_grid)
T_fem = np.full(N_NODES, T_INIT)
T_fem[0] = T_LEFT
t_current = 0.0
next_col = 0

fem_checkpoints = {}
if abs(t_lin[0] - 0.0) < 1e-9:
    fem_checkpoints[0] = T_fem.copy()
    next_col = 1

for step in range(n_dt):
    rhs = M @ T_fem
    rhs[0] = T_LEFT
    T_fem = lu_solve(A_factored, rhs)
    t_current += dt

    while next_col < len(t_lin) and t_current >= t_lin[next_col] - 1e-6:
        fem_checkpoints[next_col] = T_fem.copy()
        next_col += 1

for col_idx, T_fem_snap in fem_checkpoints.items():
    fem_grid[col_idx, :] = np.interp(x_lin, x_nodes, T_fem_snap)

error_grid = np.abs(u_grid - fem_grid)

plt.figure(figsize=(6, 5))
cp = plt.contourf(x_grid, t_grid, error_grid, levels=50, cmap='viridis')
plt.colorbar(cp, label='|PINN - FEM| (deg C)')
plt.xlabel('x (position along rod)')
plt.ylabel('t (time)')
plt.title('Pointwise absolute error: PINN vs. FEM')
plt.show()

temp_range = T_LEFT - T_INIT
print(f"Max error:  {error_grid.max():.3f} deg C  ({100*error_grid.max()/temp_range:.2f}% of full temperature range)")
print(f"Mean error: {error_grid.mean():.3f} deg C  ({100*error_grid.mean()/temp_range:.2f}% of full temperature range)")

worst_idx = np.unravel_index(np.argmax(error_grid), error_grid.shape)
print(f"Worst error located at x={x_grid[worst_idx]:.3f}, t={t_grid[worst_idx]:.1f}s -- "
      f"if this is still large, that's where to add more collocation points or training time next.")